In [24]:
from langchain_classic.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [ ]:
model = ChatOpenAI(
    model='gpt-3.5-turbo', 
    temperature=0.7,
    max_completion_tokens=50
)

In [21]:
inputs = ["python", "java"]

chat_1 = ChatPromptTemplate.from_template(
    "suggest three {pl} books. answer only by listing"
)

chat_2 = ChatPromptTemplate.from_template(
    "suggest three {pl} projects. answer only by listing"
)

In [11]:
str_parser = StrOutputParser()

In [13]:
chain_books = chat_1 | model | str_parser

In [15]:
chain_project = chat_2 | model | str_parser

In [22]:
chain_parallel = RunnableParallel({'books':chain_books, 'projects':chain_project})
chain_parallel.batch(inputs)

[{'books': '1. "Python Crash Course" by Eric Matthes\n2. "Automate the Boring Stuff with Python" by Al Sweigart\n3. "Learning Python" by Mark Lutz',
  'projects': '1. Social media sentiment analysis tool\n2. Grocery shopping list generator\n3. Stock market prediction algorithm'},
 {'books': '1. "Effective Java" by Joshua Bloch\n2. "Head First Java" by Kathy Sierra and Bert Bates\n3. "Java: The Complete Reference" by Herbert Schildt',
  'projects': '1. A social networking platform\n2. An online shopping application\n3. A task management tool'}]

In [23]:
chain_parallel.get_graph().print_ascii()

            +-------------------------------+              
            | Parallel<books,projects>Input |              
            +-------------------------------+              
                   ***               ***                   
                ***                     ***                
              **                           **              
+--------------------+              +--------------------+ 
| ChatPromptTemplate |              | ChatPromptTemplate | 
+--------------------+              +--------------------+ 
           *                                   *           
           *                                   *           
           *                                   *           
    +------------+                      +------------+     
    | ChatOpenAI |                      | ChatOpenAI |     
    +------------+                      +------------+     
           *                                   *           
           *                            

RUNNABLE PARALLEL + RUNNABLE

In [37]:
chat_3 = ChatPromptTemplate.from_template(
    "how long would it take to complete {books} and {projects}"
)

chain = chain_parallel | {'books':RunnablePassthrough(), 'projects':RunnablePassthrough()} | chat_3 | model
stream = chain.stream({'pl':'python'})

chain.get_graph().print_ascii()

            +-------------------------------+              
            | Parallel<books,projects>Input |              
            +-------------------------------+              
                   ***               ***                   
                ***                     ***                
              **                           **              
+--------------------+              +--------------------+ 
| ChatPromptTemplate |              | ChatPromptTemplate | 
+--------------------+              +--------------------+ 
           *                                   *           
           *                                   *           
           *                                   *           
    +------------+                      +------------+     
    | ChatOpenAI |                      | ChatOpenAI |     
    +------------+                      +------------+     
           *                                   *           
           *                            